## < Problem 1 >
![problem 1](https://github.com/seoupark1/Stochastic-Control/blob/main/assets/images/kinematics/quest_method/quest_method_01.png?raw=true)

In [ ]:
import numpy as np
import sympy as sp

def tilde(v):
    v = np.array(v).flatten()
    result = np.array([[0, -v[2], v[1]],
                       [v[2], 0, -v[0]],
                       [-v[1], v[0], 0]])

    return result

# singularity를 피하기 위해 rotation matrix를 적용 후, 값을 업데이트하기 위한 함수
def QUEST_parameters(B):
    Z = np.array([[B[1,2] - B[2,1]],
                 [B[2,0] - B[0,2]],
                 [B[0,1] - B[1,0]]])
    sigma = np.trace(B)
    S = B + np.transpose(B)

    return Z, sigma, S

def QUEST(v1_est, v2_est, v1_true, v2_true):
    # normalization
    v1_est = v1_est / np.linalg.norm(v1_est)
    v2_est = v2_est / np.linalg.norm(v2_est)
    v1_true = v1_true / np.linalg.norm(v1_true)
    v2_true = v2_true / np.linalg.norm(v2_true)

    # sensor's weight
    w1 = 1
    w2 = 1

    # initial value
    B = w1 * v1_est @ np.transpose(v1_true) + w2 * v2_est @ np.transpose(v2_true)
    Z, sigma, S = QUEST_parameters(B)
    K = np.block([[sigma, np.transpose(Z)], [Z, S - sigma * np.eye(3)]])

    # finding lambda_opt
    lam = sp.Symbol('lambda')
    matrix_eq = K - lam * sp.eye(4)
    f_lam = matrix_eq.det()
    f_lam_dot = f_lam.diff(lam)

    f = sp.lambdify(lam, f_lam, 'numpy')
    f_dot = sp.lambdify(lam, f_lam_dot, 'numpy')

    # Newton-Laphson method. lamba_3까지만 찾기로 하자
    lam_0 = w1 + w2
    lam_1 = lam_0 - f(lam_0) / f_dot(lam_0)
    lam_2 = lam_1 - f(lam_1) / f_dot(lam_1)
    lam_3 = lam_2 - f(lam_2) / f_dot(lam_2)
    lam_opt = lam_3

    # rotation axis & matrix for second body frame
    B_tr_max = 0
    B_tr_index = 0

    for k in range(3):
        if B[k,k] > B_tr_max:
            B_tr_max = B[k,k]
            B_tr_index = k

    if B_tr_index == 0:
        rot_matrix = np.array([[1, 0, 0],
                               [0, -1, 0],
                               [0, 0, -1]])
    elif B_tr_index == 1:
        rot_matrix = np.array([[-1, 0, 0],
                              [0, 1, 0],
                              [0, 0, -1]])
    elif B_tr_index == 2:
        rot_matrix = np.array([[-1, 0, 0],
                              [0, -1, 0],
                              [0, 0, 1]])

    # considering singularity and rotate 180(deg) about the axis
    gamma = lam_opt + sigma
    if gamma < 1:
        B = rot_matrix @ B

        # update parameters
        Z, sigma, S = QUEST_parameters(B)

    # derive qbar, dcm_BbarN
    qbar = np.linalg.inv(((lam_opt + sigma) * np.eye(3) - S)) @ Z

    dcm_BbarN = ((1 - np.vdot(qbar, qbar)) * np.eye(3) + 2 * qbar @ np.transpose(qbar) - 2 * tilde(qbar)) / (1 + np.vdot(qbar, qbar))

    # second body frame 다시 제자리로
    if gamma < 1:
        dcm_BbarN = np.linalg.inv(rot_matrix) @ dcm_BbarN

    return dcm_BbarN

v1_est = np.array([0.8273, 0.5541, -0.0920]).reshape(3,1)
v2_est = np.array([-0.8285, 0.5522, -0.0955]).reshape(3,1)
v1_true = np.array([-0.1517, -0.9669, 0.2050]).reshape(3,1)
v2_true = np.array([-0.8393, 0.4494, -0.3044]).reshape(3,1)
print(QUEST(v1_est, v2_est, v1_true, v2_true))


[[ 0.41593638 -0.85489353  0.31008705]
 [-0.83375667 -0.49463677 -0.24532483]
 [ 0.36310707 -0.15649762 -0.91851062]]
